# Incremental Processing: Snapshots as Checkpoints

A gold table that refreshes by re-reading all silver data scales linearly with total silver size. For IoT systems where silver grows indefinitely, that's a problem. This notebook shows how to use Iceberg snapshots as reliable checkpoints — processing only the new data since the last run.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import json
import shutil
from pathlib import Path
import pyarrow as pa
import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog
from pyspark.sql import functions as F

sys.path.insert(0, str(Path.cwd()))

In [ ]:
for d in ["../data/warehouse_silver", "../data/warehouse_gold"]:
    shutil.rmtree(d, ignore_errors=True)

catalog = SqlCatalog("silver", **{
    "uri": "sqlite:///../data/warehouse_silver/silver_catalog.db",
    "warehouse": "file://" + str(Path("../data/warehouse_silver").resolve()),
})
catalog.create_namespace_if_not_exists("silver")

# Load all events; split into three batches to simulate incremental arrivals
all_events = [json.loads(l) for l in open("../data/input/events.jsonl")]
batch_size = len(all_events) // 3
batches = [
    all_events[:batch_size],
    all_events[batch_size:2*batch_size],
    all_events[2*batch_size:],
]

# Write batch 1 to silver
arrow_batch1 = pa.Table.from_pandas(pd.DataFrame(batches[0]))
silver_events = catalog.create_table("silver.events", schema=arrow_batch1.schema)
silver_events.append(arrow_batch1)
snapshot_1 = silver_events.current_snapshot().snapshot_id
print(f"Batch 1: {len(batches[0]):,} events, snapshot_id={snapshot_1}")

In [ ]:
from helpers import get_spark

spark = get_spark("IncrementalProcessing", gold_warehouse="../data/warehouse_gold")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")
print("Spark ready.")

## Why Full Refresh Doesn't Scale

If we recompute the entire gold table from scratch each run:

- Run 1: 400K events → fast
- Run 2: 800K events → 2× slower
- Run 3: 1.2M events → 3× slower
- After 1 year: hundreds of millions of events → minutes or hours per run

We want the opposite: each run should take roughly the same time regardless of total silver size, processing only the delta since the last run.

## Why Timestamp-Based Checkpoints Are Fragile

The intuitive approach: `WHERE creationTime > last_run_timestamp`

Problems:

1. **Clock drift**: device clocks are often minutes or hours off from server time. An event with a 09:00 `time` field might arrive with a 14:23 `creationTime`.
2. **Precision mismatch**: `last_run_timestamp` stored as seconds, event timestamps stored as milliseconds → off-by-one-second boundary bugs.
3. **Re-runs produce different results**: the timestamp window is not deterministic across re-runs.
4. **Late data is silently dropped**: events arriving after the window closed are never picked up.
5. **Timezone confusion**: if Cumulocity stores times in UTC and your checkpoint is in local time, you may lose or double-count an hour per DST transition.

Timestamp-based incremental pipelines work well for simple cases but fail in subtle ways under IoT conditions. Iceberg gives us a better primitive: the snapshot ID.

## Iceberg Snapshots as Checkpoints

- Every Iceberg write (append, overwrite, delete) creates a new snapshot with a monotonically increasing `snapshot_id`.
- Snapshot IDs are assigned by the catalog at commit time — they are never reused and always increase.
- **Incremental read**: `table.scan(from_snapshot_id=last_processed_snapshot_id)` returns only records added in snapshots *after* that ID.
- Spark equivalent: `spark.read.format("iceberg").option("start-snapshot-id", last_snap_id).load(path)`
- This is exact, reproducible, and gap-free regardless of device clocks, timezones, or re-run timing.
- If you re-run the same pipeline twice with the same checkpoint, you process the same delta — **idempotent by design**.

In [ ]:
# Full refresh: recompute gold from all silver data
def compute_gold(spark, silver_events):
    """Compute hourly gold table from silver events at the given path."""
    df = spark.createDataFrame(silver_events.scan().to_arrow().to_pandas())
    return df.withColumn("event_hour", F.date_trunc("hour", F.to_timestamp("time"))) \
             .groupBy("event_hour", "type") \
             .agg(F.count("*").alias("event_count")) \
             .orderBy("event_hour", "type")

gold_df = compute_gold(spark, silver_events)
gold_df.writeTo("local.gold.hourly_events").createOrReplace()
last_processed_snapshot = snapshot_1  # checkpoint: the snapshot we just processed
print(f"Full refresh complete. Gold rows: {gold_df.count():,}")
print(f"Checkpoint stored: snapshot_id={last_processed_snapshot}")

## Incremental Run: Process Only New Snapshots

In [ ]:
import pyarrow as pa

# Simulate new data arriving: append batch 2
arrow_batch2 = pa.Table.from_pandas(pd.DataFrame(batches[1]))
silver_events.append(arrow_batch2)
snapshot_2 = silver_events.current_snapshot().snapshot_id
print(f"Batch 2 appended: {len(batches[1]):,} events, snapshot_id={snapshot_2}")

# Incremental read: get only the new batch data.
# In production with Spark + Iceberg: use
#   spark.read.format("iceberg").option("start-snapshot-id", last_processed_snapshot).load(table_path)
# PyIceberg 0.11 does not expose a native incremental scan on the Python API;
# we pass the batch data directly to keep the demo runnable.
# The snapshot_id checkpoint concept is identical regardless.
new_rows_arrow = arrow_batch2
print(f"New records to process: {len(new_rows_arrow):,}")

In [ ]:
# Convert incremental data to Spark DataFrame
new_df = spark.createDataFrame(new_rows_arrow.to_pandas())

# Compute gold for the new rows only
new_gold = new_df.withColumn("event_hour", F.date_trunc("hour", F.to_timestamp("time"))) \
                 .groupBy("event_hour", "type") \
                 .agg(F.count("*").alias("event_count"))

# Register for MERGE INTO
new_gold.createOrReplaceTempView("new_gold_delta")

# MERGE INTO existing gold: add new counts to existing hours, insert new hours
spark.sql("""
    MERGE INTO local.gold.hourly_events AS target
    USING new_gold_delta AS source
    ON target.event_hour = source.event_hour AND target.type = source.type
    WHEN MATCHED THEN UPDATE SET target.event_count = target.event_count + source.event_count
    WHEN NOT MATCHED THEN INSERT *
""")

# Advance checkpoint
last_processed_snapshot = snapshot_2
print(f"Incremental run complete. Checkpoint advanced to: {last_processed_snapshot}")
print(f"Gold rows now: {spark.table('local.gold.hourly_events').count():,}")

In [ ]:
# After incremental run 1: verify against a full recompute of batch 1 only
full_recompute = compute_gold(spark, silver_events)

incremental_count = spark.table("local.gold.hourly_events").count()
full_count = full_recompute.count()
print(f"Incremental gold rows: {incremental_count:,}")
print(f"Full recompute rows:    {full_count:,}")

incr_total = spark.sql("SELECT SUM(event_count) as total FROM local.gold.hourly_events").collect()[0]["total"]
full_total = full_recompute.agg(F.sum("event_count")).collect()[0][0]
print(f"Incremental total events: {incr_total:,}")
print(f"Full recompute total:     {full_total:,}")
print(f"Match (expected True): {incr_total == full_total}")

## Late Data: What It Is and Why IoT Has a Lot of It

**Definition**: an event arrives in the system *after* the time window it belongs to has already been processed.

**IoT-specific causes:**

1. Device goes offline (factory floor, field equipment, poor cellular coverage)
2. Events buffer locally on the device; reconnection triggers a large delayed upload
3. Clock drift: the device's clock is wrong; the event's `time` field is hours or days in the past
4. Edge-to-cloud aggregation delay: a gateway batches events and uploads them in bulk
5. Real example: 24-hour backlogs after connectivity outages on mobile equipment

**Impact**: an hourly aggregate for 09:00 may change hours later when late events for that hour finally arrive.

**Why Iceberg helps**: snapshot-based reads are deterministic. We can detect affected time windows from the late data and recompute only those windows.

In [ ]:
import datetime

# Simulate batch 3 with late data: some events have timestamps from 2 hours ago
# In real IoT: device was offline, reconnected, and uploaded its buffer
late_events = batches[2].copy()

# Inject 500 "late" events — they have old timestamps (simulating clock drift or delayed upload)
late_batch = []
for i, evt in enumerate(late_events[:500]):
    evt = dict(evt)
    # Override time to be 3 hours earlier than the event's actual creationTime
    if evt.get("time"):
        try:
            t = datetime.datetime.fromisoformat(evt["time"].replace("Z", "+00:00"))
            t_late = t - datetime.timedelta(hours=3)
            evt["time"] = t_late.isoformat().replace("+00:00", "Z")
        except Exception:
            pass
    late_batch.append(evt)

# Append batch 3 (including the late events) to silver
all_batch3 = late_batch + late_events[500:]
arrow_batch3 = pa.Table.from_pandas(pd.DataFrame(all_batch3))
silver_events.append(arrow_batch3)
snapshot_3 = silver_events.current_snapshot().snapshot_id
print(f"Batch 3 (with {len(late_batch)} late events): {len(all_batch3):,} total, snapshot_id={snapshot_3}")

In [ ]:
# Incremental data: batch 3 (including late events)
new_rows3 = arrow_batch3
new_df3 = spark.createDataFrame(new_rows3.to_pandas())
new_df3 = new_df3.withColumn("event_hour", F.date_trunc("hour", F.to_timestamp("time")))

# Find which gold windows are affected (late events touch earlier time windows)
affected_hours = new_df3.select("event_hour", "type").distinct()
print(f"Affected gold windows in batch 3: {affected_hours.count():,}")
affected_hours.orderBy("event_hour").show(10)

In [ ]:
# For affected windows: recompute from all silver data, not just the delta
# This handles both late data and corrections

# Get the full silver dataset (all snapshots) via Arrow conversion
all_silver = spark.createDataFrame(silver_events.scan().to_arrow().to_pandas()).withColumn(
    "event_hour", F.date_trunc("hour", F.to_timestamp("time"))
)

# Collect affected hours as a list for the filter
affected_list = [row["event_hour"] for row in affected_hours.select("event_hour").distinct().collect()]

# Recompute only the affected time windows
recomputed_windows = all_silver.filter(F.col("event_hour").isin(affected_list)) \
    .groupBy("event_hour", "type") \
    .agg(F.count("*").alias("event_count"))

# For non-late windows in batch 3, just do incremental merge
non_late_windows = new_df3.filter(~F.col("event_hour").isin(affected_list)) \
    .groupBy("event_hour", "type") \
    .agg(F.count("*").alias("event_count"))

# Combine
all_updates = recomputed_windows.union(non_late_windows)
all_updates.createOrReplaceTempView("all_updates")

spark.sql("""
    MERGE INTO local.gold.hourly_events AS target
    USING all_updates AS source
    ON target.event_hour = source.event_hour AND target.type = source.type
    WHEN MATCHED THEN UPDATE SET target.event_count = source.event_count
    WHEN NOT MATCHED THEN INSERT *
""")

last_processed_snapshot = snapshot_3
print(f"Late-data-aware run complete. Checkpoint: {last_processed_snapshot}")

In [ ]:
# Verify against full recompute
final_full = compute_gold(spark, silver_events)
final_incr_total = spark.sql("SELECT SUM(event_count) FROM local.gold.hourly_events").collect()[0][0]
final_full_total = final_full.agg(F.sum("event_count")).collect()[0][0]
print(f"Incremental (late-aware) total: {final_incr_total:,}")
print(f"Full recompute total:            {final_full_total:,}")
print(f"Match: {final_incr_total == final_full_total}")

## Checkpoint Storage

In these notebooks we track `last_processed_snapshot` as an in-memory variable. In production:

- Store it in a small Iceberg table (most durable option — queryable, versioned)
- Or in a JSON file on object storage (simpler but not queryable)
- Or in Airflow XComs / workflow state (if using an orchestrator)

Never use a mutable variable — restarts and failures must be able to recover the last good checkpoint.

## Review Questions

1. What guarantees does snapshot-based incremental processing provide that `WHERE creationTime > last_run_time` does not?
2. A device goes offline for 3 days. When it reconnects and uploads its backlog, which gold time windows are affected? How does your pipeline detect and handle this?
3. Why is snapshot-based incremental processing idempotent? What happens if you run the same incremental step twice?
4. A watermark defines a "late arrival deadline" — events arriving after the deadline are discarded. What is the trade-off between a short watermark (tight deadline) and a long one?
5. Your incremental pipeline runs every 15 minutes. The silver table receives 50 batches per day. How many snapshots will the silver table have after one week? What does this mean for snapshot expiration policy?

## Challenges

- Implement a `PipelineCheckpoint` class that stores the last processed snapshot ID in a JSON file and loads it on restart. Verify that re-running the pipeline from a fresh kernel (loading the stored checkpoint) produces the same gold output.
- Implement a timestamp-based incremental pipeline and deliberately trigger the late-data bug: process batch 2, advance the timestamp checkpoint, then append batch 3 with late events. Show that some late events are silently dropped.
- Extend the late-data handler: instead of reprocessing the affected windows inline, create a separate "correction" gold table that records each recomputed window with a `recomputed_at` timestamp. This creates an audit trail of corrections.

## Summary

Key takeaways:

- **Iceberg snapshot IDs** are monotonically increasing, immutable checkpoints. `scan(from_snapshot_id=...)` reads only what changed — no clock drift, no gaps, no duplicates from re-runs.
- **Late IoT data** (from buffered devices, clock drift, or gateway delays) can affect already-processed time windows. Detect it by checking which gold windows are touched by the new snapshot data, then recompute those windows from the full silver history.
- **Snapshot-based checkpoints are idempotent**: running the same pipeline step twice with the same checkpoint produces the same result.
- In production, store the checkpoint in a durable, queryable location (another Iceberg table or object-store JSON). Never rely on an in-memory variable.
- **Next**: ensuring the quality of gold data before it reaches consumers.